In [1]:
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import os, sys
from tqdm import tqdm
import dask
import h5py
import time
from dask.distributed import Client, LocalCluster

# from huggingface_hub import snapshot_download

DATA_PATH = os.path.join(os.getenv("AGED"), "CBMrepositoryData/perturbational_data")
# DATA_PATH = os.path.join(os.getenv("MLAB"), "projects/brcameta/projects/sig_recon/data/tahoe")
STACK_PATH = os.path.join(os.getenv("MLAB"), "projects/brcameta/projects/sig_recon/scripts/stack")

In [25]:
import re
def normalize_drug_name(drug_name):
    """
    Drug names have slight differences in pseudobulk and single cell datasets.
    """
    drug_name = str(drug_name).strip()
    
    # Specific cases
    manual_mappings = {
        'Dapagliflozin ((2S)-1,2-propanediol, hydrate)': 'Dapagliflozin 2S-1'
    }
    
    if drug_name in manual_mappings:
        drug_name = manual_mappings[drug_name]
        
    # Remove parentheses and their contents if it's a salt/form
    drug_name = re.sub(r'\(([^)]+)\)', r'\1', drug_name)
        
    return drug_name


In [2]:
# 1. Writing Prompt/Query Data
use_gpu = False

if use_gpu:
    import rapids_singlecell as rsc
    SPARSE_CHUNK_SIZE = 100_000
    from dask_cuda import LocalCUDACluster

    import rmm
    import cupy as cp
    from cupyx.scipy import sparse as spx

    from rmm.allocators.cupy import rmm_cupy_allocator

    def set_mem():
        rmm.reinitialize(managed_memory=True)
        cp.cuda.set_allocator(rmm_cupy_allocator)
    gpus = "0,1" # comma separated like "0,1,2" for 3 gpus
    cluster = LocalCUDACluster(CUDA_VISIBLE_DEVICES=gpus)
    dask.array.register_chunk_type(spx.csr_matrix)
else:
    SPARSE_CHUNK_SIZE = 100_000
    cluster = LocalCluster(n_workers=20)

client = Client(cluster)

with h5py.File(os.path.join(DATA_PATH, f"tahoe/h5ad/tahoe_plate_merged_cpu.h5ad"), "r") as f:
    adata = ad.AnnData(
        obs=ad.io.read_elem(f["obs"]),
        var=ad.io.read_elem(f["var"]),
    )
    adata.X = ad.experimental.read_elem_as_dask(
        f["X"], chunks=(SPARSE_CHUNK_SIZE, adata.shape[1])
    )

In [3]:
adata

AnnData object with n_obs × n_vars = 95624334 × 15625
    obs: 'sample', 'gene_count', 'tscp_count', 'mread_count', 'drugname_drugconc', 'drug', 'cell_line', 'sublibrary', 'BARCODE', 'pcnt_mito', 'S_score', 'G2M_score', 'phase', 'pass_filter', 'cell_name', 'plate'

In [ ]:
adata.obs

In [ ]:
# Finding out which cell_names correspond to which cell_lines
cell_lines = ["CVCL_0023", "CVCL_0028", "CVCL_0099", "CVCL_0131", "CVCL_0152", "CVCL_0293", "CVCL_0334", "CVCL_0366", "CVCL_0371", "CVCL_0397"]

In [ ]:
cell_line_name_counts = adata.obs.loc[adata.obs['cell_line'].isin(cell_lines),["cell_line", "cell_name"]].value_counts()

In [ ]:
mapping = adata.obs[['cell_line', 'cell_name']].drop_duplicates().set_index('cell_line')['cell_name'].to_dict()
cell_names = [mapping[cl] for cl in cell_lines if cl in mapping]

print(cell_names)

In [ ]:
    np.random.seed(42)
    cell_names = adata.obs['cell_line'].unique()
    np.random.shuffle(cell_names)
    cell_names = cell_names[:10]
    print(f"\nCell lines selected: {cell_names}")

In [ ]:
import ast
adata.obs['drugconc'] = adata.obs['drugname_drugconc'].apply(
    lambda x: ast.literal_eval(x)[0][1]
)

In [8]:
import pandas as pd
drug_splits = pd.read_csv(os.path.join(os.getenv("MLAB"), "projects/brcameta/projects/sig_recon/data/sigs/tahoe/drug_splits.csv"))

In [9]:
shared_drugs = set(drug_splits['drug'].values)
adata_drugs = set(adata.obs['drug'].unique())

# Check if all shared drugs are in adata
all_present = shared_drugs.issubset(adata_drugs)
print(f"All shared drugs present in adata: {all_present}")
# Find missing drugs
missing_drugs = shared_drugs - adata_drugs
print(f"Missing drugs ({len(missing_drugs)}): {missing_drugs}")
# Find extra drugs in adata not in splits
extra_drugs = adata_drugs - shared_drugs
print(f"Extra drugs in adata ({len(extra_drugs)}): {extra_drugs}")

NameError: name 'adata' is not defined

In [23]:
import re

def normalize_drug_name(drug_name):
    """
    More comprehensive normalization:
    - Remove parentheses
    - Normalize Unicode characters
    - Standardize whitespace
    - Handle special cases
    """
    drug_name = str(drug_name).strip()
        # Specific cases
    manual_mappings = {
        'Dapagliflozin ((2S)-1,2-propanediol, hydrate)': 'Dapagliflozin 2S-1'
    }
    
    if drug_name in manual_mappings:
        drug_name = manual_mappings[drug_name]
        
    # Remove parentheses and their contents if it's a salt/form
    drug_name = re.sub(r'\(([^)]+)\)', r'\1', drug_name)
        
    return drug_name

In [25]:
# Apply to both datasets
drug_splits['drug_normalized'] = drug_splits['drug'].apply(normalize_drug_name)
adata.obs['drug_normalized'] = adata.obs['drug'].apply(normalize_drug_name)

# Now check
shared_drugs_normalized = set(drug_splits['drug_normalized'].values)
adata_drugs_normalized = set(adata.obs['drug_normalized'].unique())

all_present = shared_drugs_normalized.issubset(adata_drugs_normalized)
print(f"All shared drugs present after normalization: {all_present}")

missing = shared_drugs_normalized - adata_drugs_normalized
print(f"Missing drugs: {len(missing)}")
if missing:
    print(f"Still missing: {missing}")

All shared drugs present after normalization: True
Missing drugs: 0


In [29]:
len(adata.obs.drug.unique())

380

In [2]:
base_adata_ctrl = sc.read("/restricted/projectnb/agedisease/CBMrepositoryData/perturbational_data/tahoe/stack_subsets/ctrl_comparison/NCI-H23_maxconc_dmso.h5ad")

In [26]:
base_adata_ctrl.obs['drug'] = base_adata_ctrl.obs['drug'].apply(normalize_drug_name)

In [27]:
unique_pbs = set(base_adata_ctrl.obs.drug.unique())

In [28]:
import os
dir_path = "/restricted/projectnb/agedisease/CBMrepositoryData/perturbational_data/tahoe/stack_pred/ctrl_generation/NCI-H23/"
done_drugs = set([
    os.path.splitext(f)[0] 
    for f in os.listdir(dir_path) 
    if os.path.isfile(os.path.join(dir_path, f))
])

In [29]:
len(done_drugs)

81

In [34]:
len(unique_pbs)

111

In [33]:
unique_pbs

{'18β-Glycyrrhetinic acid',
 '8-Hydroxyquinoline',
 'Acetohexamide',
 'Adenine',
 'Adenosine',
 'Afatinib',
 'Aliskiren',
 'Allantoin',
 'Almonertinib hydrochloride',
 'Almonertinib mesylate',
 'Amsacrine',
 'Anethole trithione',
 'Artesunate',
 'Azithromycin hydrate',
 'Baicalin',
 'Balsalazide sodium hydrate',
 'Belumosudil',
 'Berbamine',
 'Berbamine dihydrochloride',
 'Berberine chloride hydrate',
 'Bergenin',
 'Bimatoprost',
 'Binimetinib',
 'Busulfan',
 'Cepharanthine',
 'Chlorhexidine diacetate',
 'Ciclopirox',
 'Clonidine hydrochloride',
 'Clotrimazole',
 'Cloxacillin sodium',
 'Cysteamine hydrochloride',
 'Cytarabine',
 'DMSO_TF',
 'Daidzin',
 'Dapagliflozin 2S-1',
 'Darinaparsin',
 'Demeclocycline',
 'Diammonium Glycyrrhizinate',
 'Docetaxel',
 'Elagolix sodium',
 'Everolimus',
 'Flumatinib mesylate',
 'Fluvoxamine maleate',
 'Folic acid',
 'Fumaric acid',
 'Furosemide',
 'Fusidic acid',
 'Gallic acid',
 'Gallic acid hydrate',
 'Gemcitabine',
 'Gemfibrozil',
 'Goserelin aceta

In [32]:
done_drugs

{'18_-Glycyrrhetinic_acid',
 '8-Hydroxyquinoline',
 'Acetohexamide',
 'Adenine',
 'Adenosine',
 'Afatinib',
 'Aliskiren',
 'Allantoin',
 'Almonertinib_hydrochloride',
 'Almonertinib_mesylate',
 'Amsacrine',
 'Anethole_trithione',
 'Artesunate',
 'Azithromycin_hydrate',
 'Baicalin',
 'Balsalazide_sodium_hydrate',
 'Belumosudil',
 'Berbamine',
 'Berbamine_dihydrochloride',
 'Berberine_chloride_hydrate',
 'Bergenin',
 'Bimatoprost',
 'Binimetinib',
 'Busulfan',
 'Cepharanthine',
 'Chlorhexidine_diacetate',
 'Ciclopirox',
 'Clonidine_hydrochloride',
 'Clotrimazole',
 'Cloxacillin_sodium',
 'Cysteamine_hydrochloride',
 'Cytarabine',
 'DMSO_TF',
 'Daidzin',
 'Dapagliflozin_2S-1',
 'Darinaparsin',
 'Demeclocycline',
 'Diammonium_Glycyrrhizinate',
 'Docetaxel',
 'Elagolix_sodium',
 'Everolimus',
 'Flumatinib_mesylate',
 'Fluvoxamine_maleate',
 'Folic_acid',
 'Fumaric_acid',
 'Furosemide',
 'Fusidic_acid',
 'Gallic_acid',
 'Gallic_acid_hydrate',
 'Gemcitabine',
 'Gemfibrozil',
 'Goserelin_aceta

In [30]:
unique_pbs - done_drugs

{'18β-Glycyrrhetinic acid',
 'Almonertinib hydrochloride',
 'Almonertinib mesylate',
 'Anethole trithione',
 'Azithromycin hydrate',
 'Balsalazide sodium hydrate',
 'Berbamine dihydrochloride',
 'Berberine chloride hydrate',
 'Chlorhexidine diacetate',
 'Clonidine hydrochloride',
 'Cloxacillin sodium',
 'Cysteamine hydrochloride',
 'Dapagliflozin 2S-1',
 'Diammonium Glycyrrhizinate',
 'Elagolix sodium',
 'Flumatinib mesylate',
 'Fluvoxamine maleate',
 'Folic acid',
 'Fumaric acid',
 'Fusidic acid',
 'Gallic acid',
 'Gallic acid hydrate',
 'Goserelin acetate',
 'Idarubicin hydrochloride',
 'Irinotecan hydrochloride',
 'Lenalidomide hemihydrate',
 'Lidocaine hydrochloride',
 'Lipoic acid',
 'Lumateperone tosylate',
 'Medroxyprogesterone acetate',
 'Mitoxantrone dihydrochloride',
 'Neratinib maleate',
 'Norepinephrine hydrochloride',
 'Palmatine chloride',
 'Pasireotide acetate',
 'Pentamidine isethionate',
 'Pravastatin sodium',
 'Pyridoxine hydrochloride',
 'R-Verapamil hydrochloride',


In [5]:
len(remaining_pbs)

15

In [6]:
base_adata_ctrl.obs.cell_name.value_counts()[base_adata_ctrl.obs.cell_name.value_counts() > 0]

cell_name
NCI-H23    354057
Name: count, dtype: int64

In [7]:
base_adata_ctrl.obs.drug.value_counts()[base_adata_ctrl.obs.drug.value_counts() > 0]

drug
DMSO_TF                       43557
Clonidine hydrochloride        4499
Berberine chloride hydrate     4458
Pentagastrin                   4284
Artesunate                     4209
                              ...  
Gemcitabine                    1284
Belumosudil                    1171
Dapagliflozin 2S-1              803
Idarubicin hydrochloride        578
Homoharringtonine               544
Name: count, Length: 111, dtype: int64

In [4]:
base_adata_10 = sc.read("/restricted/projectnb/agedisease/CBMrepositoryData/perturbational_data/tahoe/stack_subsets/10th_splits/NCI-H23_1_10th_split_2.h5ad")

In [5]:
base_adata_10.obs.drug.value_counts()[base_adata_10.obs.drug.value_counts() > 0]

drug
DMSO_TF                     43557
Baicalin                    38776
Quinestrol                  35425
Triclosan                   32121
Resveratrol                 31145
                            ...  
Gemcitabine                  1284
Belumosudil                  1171
Dapagliflozin 2S-1            803
Idarubicin hydrochloride      578
Homoharringtonine             544
Name: count, Length: 111, dtype: int64

In [12]:
base_adata_10.obs.cell_name.value_counts()[base_adata_10.obs.cell_name.value_counts() > 0]

cell_name
NCI-H23       354057
LoVo           56849
HEC-1-A        48849
A498           42197
SNU-423        27633
Panc 03.27     27231
MIA PaCa-2     24958
HCT15          24845
SW48           18209
SNU-1          16408
Name: count, dtype: int64

In [6]:
base_adata_10.obs[base_adata_10.obs.cell_name != "NCI-H23"].drug.value_counts()[base_adata_10.obs[base_adata_10.obs.cell_name != "NCI-H23"].drug.value_counts() > 0]

drug
Baicalin               34672
Quinestrol             31577
Triclosan              28694
Resveratrol            27658
Cytarabine             25781
Folic acid             25249
Regorafenib            25176
Fluvoxamine maleate    23985
Tofacitinib citrate    23796
Radotinib              20623
Demeclocycline         19968
Name: count, dtype: int64

In [7]:
base_adata_90 = sc.read("/restricted/projectnb/agedisease/CBMrepositoryData/perturbational_data/tahoe/stack_subsets/90th_splits/NCI-H23_9_10th_split_1.h5ad")

In [8]:
base_adata_90.obs.drug.value_counts()[base_adata_90.obs.drug.value_counts() > 0]

drug
Trametinib                    47688
Clonidine hydrochloride       43854
Berberine chloride hydrate    43774
DMSO_TF                       43557
Pentagastrin                  40353
                              ...  
Vinblastine sulfate            2213
Almonertinib hydrochloride     2192
Saquinavir                     1983
Selinexor                      1826
Larotrectinib                  1789
Name: count, Length: 111, dtype: int64

In [13]:
base_adata_90.obs[base_adata_90.obs.cell_name != "NCI-H23"].drug.value_counts()[base_adata_90.obs[base_adata_90.obs.cell_name != "NCI-H23"].drug.value_counts() > 0]

drug
Trametinib                    44428
Clonidine hydrochloride       39355
Berberine chloride hydrate    39316
Binimetinib                   36745
Afatinib                      36439
                              ...  
Belumosudil                   17289
Cepharanthine                 15837
Idarubicin hydrochloride      14644
Gemcitabine                   11386
Dapagliflozin 2S-1             9178
Name: count, Length: 99, dtype: int64

In [8]:
test_adata = sc.read("/restricted/projectnb/agedisease/CBMrepositoryData/perturbational_data/tahoe/stack_subsets/ctrl_comparison/NCI-H23_other_dmso.h5ad")

In [9]:
test_adata.obs.drug.value_counts()[test_adata.obs.drug.value_counts() > 0]

drug
DMSO_TF    403167
Name: count, dtype: int64

Tahoe splits and stack model don't share many genes. Unclear why. We are subsetting Tahoe's 60k features to 15k based on union of HVGs (5000) across 14 plates. There are some ENSEMBL IDs, mostly novel transcripts and lncRNAs that don't have a matching gene symbol. Yet even with the remaining 12k HGNC symbols, only 3.4k match with stack.

In [ ]:
import pickle

file_path = '/restricted/projectnb/brcameta/projects/sig_recon/scripts/stack/notebooks/tutorial-pred-model/basecount_1000per_15000max.pkl' # Replace with your file's path

# Open the file in binary read mode ('rb')
with open(file_path, 'rb') as file:
    # Deserialize the data from the file
    genelist = pickle.load(file)

In [ ]:
len(set(np.array(genelist).tolist()) & set(test_adata.var_names))

In [ ]:
len(set(np.array(genelist).tolist()) & set(base_adata.var_names))

In [ ]:
matches = [gene for gene in base_adata.var_names if 'AS1' in gene]
print(f"Matching genes: {matches}")

In [ ]:
gene_ids = base_adata.var_names
ensembl_mask = [str(g).startswith("ENSG") for g in gene_ids]
ensembl_ids = [gene_ids[i] for i, mask in enumerate(ensembl_mask) if mask]

In [ ]:
print(f"Found {len(ensembl_ids)} Ensembl IDs out of {len(gene_ids)} total genes")

In [ ]:
import mygene

mg = mygene.MyGeneInfo()
results = mg.querymany(
    ensembl_ids,
    scopes='ensembl.gene',
    fields='symbol',
    species='human',
    returnall=True,
    verbose=False
)